# 07 — Evaluate All Models on Adverse Conditions

Evaluates all 4 models on every adverse condition split discovered during
data preparation (loaded dynamically from `configs/conditions.json`).

Robustness metrics (mAP drop, retention %) are computed relative to the
clear-day baseline loaded from `results/clear_day/scores.json`.

**Prerequisites:** notebook 06 must have completed successfully.

In [ ]:
from pathlib import Path
import json, torch
from dotenv import load_dotenv
from src.eval_utils import compute_map, compute_precision_recall, compute_robustness_metrics
from src.train_utils import setup_logging

DRIVE_ROOT   = Path('/content/drive/MyDrive/FON/master_rad/')
CONFIGS_DIR  = Path('/content/data_prepared/configs')
CHECKPOINTS  = DRIVE_ROOT / 'checkpoints'
RESULTS_ROOT = DRIVE_ROOT / 'results'

load_dotenv(DRIVE_ROOT / '.env')
logger = setup_logging()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

conditions = json.loads((CONFIGS_DIR / 'conditions.json').read_text())
ADVERSE_CONDITIONS = conditions['adverse']
print(f'Loaded {len(ADVERSE_CONDITIONS)} adverse conditions:', ADVERSE_CONDITIONS)

clear_scores = json.loads((RESULTS_ROOT / 'clear_day' / 'scores.json').read_text())
print('Loaded clear-day baselines for:', list(clear_scores.keys()))

In [ ]:
from ultralytics import YOLO, RTDETR

yolov11 = YOLO(str(CHECKPOINTS / 'yolov11/run/weights/best.pt'))
yolov12 = YOLO(str(CHECKPOINTS / 'yolov12/run/weights/best.pt'))
rtdetr  = RTDETR(str(CHECKPOINTS / 'rtdetr/run/weights/best.pt'))

# TODO: Load RF-DETR checkpoint
# from rfdetr import RFDETRMedium
# rfdetr_model = RFDETRMedium()
# rfdetr_model.load(str(CHECKPOINTS / 'rfdetr/best.pt'))

In [ ]:
for condition in ADVERSE_CONDITIONS:
    RESULTS_DIR = RESULTS_ROOT / condition
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    scores = {}
    robustness = {}

    for name, model in [('yolov11', yolov11), ('yolov12', yolov12), ('rtdetr', rtdetr)]:
        logger.info(f'Evaluating {name} on {condition}...')

        results = model.val(
            data=str(CONFIGS_DIR / 'dataset.yaml'),
            split=f'test_{condition}',
            device=DEVICE,
        )
        m = {**compute_map(results), **compute_precision_recall(results)}
        scores[name] = m

        rob = compute_robustness_metrics(
            clear_map=clear_scores[name]['map50'],
            adverse_map=m['map50'],
        )
        robustness[name] = rob
        logger.info(
            f'{name} {condition}: mAP50={m["map50"]:.4f}  '
            f'drop={rob["map_drop"]:.4f}  retention={rob["retention_pct"]:.1f}%'
        )

    # TODO: Add RF-DETR evaluation here once API is confirmed

    (RESULTS_DIR / 'scores.json').write_text(json.dumps(scores, indent=2))
    (RESULTS_DIR / 'robustness.json').write_text(json.dumps(robustness, indent=2))
    print(f'Saved {condition} results to {RESULTS_DIR}')